# Market Structure Analysis

This notebook estimates and visualises changes in S&P 500 market structure using rolling return windows.


In [86]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Iterable

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.cluster import AffinityPropagation
from sklearn.covariance import GraphicalLassoCV, graphical_lasso
from sklearn.manifold import MDS

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42

## Configuration

All paths and analysis parameters are centralised here. To run the notebook from another location or with a different dataset, update the values below.

The data is sourced from Polygon API (as shown within data_preparation.ipynb) and manipulated so that there is only the data from when the stock was within the S&P500.

In [87]:
DATA_DIR = "../data"

# I added a dataclass to store all the config options to have. Mess around with them in one place
@dataclass(frozen=True)
class AnalysisConfig:
    stock_data_dir: Path
    sector_mapping_path: Path
    output_dir: Path
    start_year: int = 2004
    end_year: int = 2024
    rolling_window_years: int = 1
    # Coverage over the WHOLE 2004-2024 panel. At 0.50 this silently deletes every
    # stock that did not survive the period - BSC (0.19), MER (0.22), WB (0.22),
    # NCC (0.22), CFC (0.19) - reintroducing the survivorship bias that the
    # index-membership filtering exists to avoid. Per-window adequacy is enforced
    # separately in calculate_returns, so this only needs to drop genuine junk.
    min_ticker_coverage: float = 0.05
    min_window_observations: int = 50
    min_window_tickers: int = 10
    graphical_lasso_edge_k: int = 2
    affinity_damping: float = 0.80
    # Calibrated once on the pooled sample and held fixed across years, so that
    # edge counts and cluster counts are comparable year to year rather than
    # reflecting a per-year change in the penalty / preference.
    graphical_lasso_alpha: float | None = None
    affinity_preference: float | None = None
    gif_duration_seconds: float = 0.80
    figure_size: tuple[int, int] = (15, 10)
    figure_dpi: int = 150



config = AnalysisConfig(

      
    # the merged panel (original pull + re-pull) when present, else the original
    stock_data_dir=(Path("../data/stock_data_filtered_for_s&p500_membership_merged")
                    if Path("../data/stock_data_filtered_for_s&p500_membership_merged").exists()
                    else Path("../data/stock_data_filtered_for_s&p500_membership")),
    sector_mapping_path=Path(DATA_DIR) / "ticker_to_sector.csv",
    output_dir=Path("../../figures").resolve()
)

Path(config.output_dir / "mds_frames").mkdir(parents=True, exist_ok=True)
Path(config.output_dir / "affinity_propagation_frames").mkdir(parents=True, exist_ok=True)

config


AnalysisConfig(stock_data_dir=PosixPath('../data/stock_data_filtered_for_s&p500_membership'), sector_mapping_path=PosixPath('../data/ticker_to_sector.csv'), output_dir=PosixPath('/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/figures'), start_year=2004, end_year=2024, rolling_window_years=1, min_ticker_coverage=0.5, min_window_observations=50, min_window_tickers=10, graphical_lasso_edge_k=2, affinity_damping=0.8, gif_duration_seconds=0.8, figure_size=(15, 10), figure_dpi=150)

## Sector Definitions

The sectors are pulled from yfinance's sector labelling which are similiar to Morningstar's broad sector classifications.

In [88]:
SECTOR_TO_ID = {
    "Technology": 0,
    "Financials": 1,
    "Healthcare": 2,
    "Energy": 3,
    "Industrials": 4,
    "Utilities": 5,
    "Consumer Discretionary": 6,
    "Consumer Staples": 7,
    "Real Estate": 8,
    "Materials": 9,
    "Communication Services": 10,
    "Unknown": 11,
}

SECTOR_NORMALISATION = {
    "Financial Services": "Financials",
    "Consumer Defensive": "Consumer Staples",
    "Consumer Cyclical": "Consumer Discretionary",
    "Basic Materials": "Materials",
    "Communication": "Communication Services",
}

SECTOR_CMAP = plt.cm.tab20
SECTOR_COLOR_DENOMINATOR = len(SECTOR_TO_ID) - 1

## Reusable Data and Plotting Functions

The helper functions below keep the modeling cells short and make path handling, data validation, and plotting behavior reusable across both methods.
Skip if needed


In [89]:
def list_stock_files(stock_data_dir: Path) -> list[Path]:
    if not stock_data_dir.exists():
        raise FileNotFoundError(f"Stock data directory does not exist: {stock_data_dir}")

    stock_files = sorted(path for path in stock_data_dir.iterdir() if path.suffix.lower() == ".csv")
    if not stock_files:
        raise FileNotFoundError(f"No CSV files found in stock data directory: {stock_data_dir}")
    return stock_files


def load_close_price_file(file_path: Path) -> pd.DataFrame:
    ticker = file_path.stem.upper()
    frame = pd.read_csv(file_path)

    required_columns = {"date", "close"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{file_path.name} is missing required columns: {sorted(missing_columns)}")

    frame = frame.loc[:, ["date", "close"]].copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame = frame.dropna(subset=["date"]).rename(columns={"close": ticker})
    return frame


def load_price_matrix(stock_data_dir: Path, min_ticker_coverage: float) -> pd.DataFrame:
    price_frames: list[pd.DataFrame] = []
    skipped_files: list[tuple[str, str]] = []

    for file_path in list_stock_files(stock_data_dir):
        try:
            price_frames.append(load_close_price_file(file_path))
        except Exception as exc:
            skipped_files.append((file_path.name, str(exc)))

    if not price_frames:
        raise ValueError("No price data loaded. Check stock_data_dir and input CSV schema.")

    price_matrix = price_frames[0]
    for frame in price_frames[1:]:
        price_matrix = price_matrix.merge(frame, on="date", how="outer")

    price_matrix = price_matrix.sort_values("date").set_index("date")
    price_matrix = price_matrix.apply(pd.to_numeric, errors="coerce")

    coverage = price_matrix.notna().mean()
    price_matrix = price_matrix.loc[:, coverage >= min_ticker_coverage]

    print(f"Loaded price matrix: {price_matrix.shape[0]:,} dates x {price_matrix.shape[1]:,} tickers")
    if skipped_files:
        print(f"Skipped {len(skipped_files):,} malformed files. First skipped file: {skipped_files[0]}")

    return price_matrix


def load_sector_mapping(sector_mapping_path: Path) -> dict[str, str]:
    if not sector_mapping_path.exists():
        print(f"Sector mapping file not found: {sector_mapping_path}. Falling back to Unknown sectors.")
        return {}

    sector_frame = pd.read_csv(sector_mapping_path)
    normalized_columns = {column.lower().strip(): column for column in sector_frame.columns}

    ticker_column = normalized_columns.get("ticker")
    sector_column = normalized_columns.get("sector")
    if ticker_column is None or sector_column is None:
        raise ValueError("Sector mapping must contain Ticker and Sector columns.")

    sector_frame = sector_frame.loc[:, [ticker_column, sector_column]].dropna().copy()
    sector_frame[ticker_column] = sector_frame[ticker_column].astype(str).str.strip().str.upper()
    sector_frame[sector_column] = sector_frame[sector_column].astype(str).str.strip().replace(SECTOR_NORMALISATION)

    ticker_to_sector = dict(zip(sector_frame[ticker_column], sector_frame[sector_column]))
    print(f"Loaded {len(ticker_to_sector):,} ticker-sector mappings")
    return ticker_to_sector


def calculate_returns(price_matrix: pd.DataFrame, min_observed_share: float = 0.50) -> pd.DataFrame:
    returns = price_matrix.pct_change(fill_method=None).dropna(how="all")
    
    returns = returns.loc[:, returns.notna().mean() > min_observed_share]
    returns = returns.fillna(0.0)
    
    returns = returns.loc[:, returns.std() > 1e-4]
    
    if not returns.empty:
        generator = np.random.default_rng(RANDOM_STATE)
        
        noise_df = pd.DataFrame(
            generator.normal(0, 1e-8, size=returns.shape),
            index=returns.index,
            columns=returns.columns
        )
        returns = returns + noise_df
        
    return returns


def standardize_returns(returns: pd.DataFrame) -> np.ndarray:
    values = returns.to_numpy(dtype=float)
    return (values - values.mean(axis=0)) / (values.std(axis=0) + 1e-9)


def create_mds_model() -> MDS:
    base_kwargs = {
        "n_components": 2,
        "dissimilarity": "precomputed",
        "random_state": RANDOM_STATE,
        "max_iter": 500,
    }
    try:
        return MDS(**base_kwargs, normalized_stress="auto")
    except TypeError:
        return MDS(**base_kwargs)


def build_mds_layout(price_matrix: pd.DataFrame, min_overlap_days: int = 60) -> dict[str, np.ndarray]:
    returns = price_matrix.pct_change(fill_method=None)
    returns = returns.loc[:, returns.std() > 1e-8]
    correlation_matrix = returns.corr(min_periods=min_overlap_days).fillna(0.0)
    distance_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - correlation_matrix)))
    coordinates = create_mds_model().fit_transform(distance_matrix)
    layout = dict(zip(correlation_matrix.index, coordinates))
    print(f"Built global MDS layout for {len(layout):,} tickers")
    return layout


def window_coverage(returns: pd.DataFrame, year: int) -> float:
    """Share of the year's trading days actually observed, for honest frame titles."""
    trading_days = len(pd.bdate_range(f"{year}-01-01", f"{year}-12-31"))
    return len(returns) / trading_days


def window_price_matrix(price_matrix: pd.DataFrame, end_year: int, rolling_window_years: int) -> tuple[int, pd.DataFrame]:
    start_year = end_year - rolling_window_years + 1
    year_mask = (price_matrix.index.year >= start_year) & (price_matrix.index.year <= end_year)
    return start_year, price_matrix.loc[year_mask]


def sector_color_ids(tickers: Iterable[str], ticker_to_sector: dict[str, str]) -> list[int]:
    return [
        SECTOR_TO_ID.get(ticker_to_sector.get(ticker, "Unknown"), SECTOR_TO_ID["Unknown"])
        for ticker in tickers
    ]


def build_sector_legend(color_ids: Iterable[int]) -> list[Line2D]:
    present_color_ids = set(color_ids)
    legend_elements = []

    for sector, sector_id in sorted(SECTOR_TO_ID.items(), key=lambda item: item[1]):
        if sector == "Unknown" and sector_id not in present_color_ids:
            continue
        legend_elements.append(
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                label=sector,
                markerfacecolor=SECTOR_CMAP(sector_id / SECTOR_COLOR_DENOMINATOR),
                markersize=10,
            )
        )
    return legend_elements

def render_network_frame(
    *,
    frame_path: Path,
    title: str,
    tickers: list[str],
    positions: np.ndarray,
    edges: np.ndarray,
    edge_strengths: np.ndarray,
    node_sizes: np.ndarray,
    ticker_to_sector: dict[str, str],
    figure_size: tuple[int, int],
    figure_dpi: int,
    node_color_ids: np.ndarray | None = None,
    legend_title: str = "Sector (yfinance)",
    max_labels: int = 40,
) -> None:
    # Default: colour by sector. Pass node_color_ids to colour by cluster instead.
    color_by_sector = node_color_ids is None
    color_ids = sector_color_ids(tickers, ticker_to_sector) if color_by_sector else list(node_color_ids)
    denominator = SECTOR_COLOR_DENOMINATOR if color_by_sector else 19
    node_colors = [SECTOR_CMAP((color_id % 20) / denominator) for color_id in color_ids]

    fig, ax = plt.subplots(figsize=figure_size)
    ax.axis("off")

    # 1. Draw Edges
    for source_idx in range(len(tickers)):
        for target_idx in range(source_idx + 1, len(tickers)):
            if not edges[source_idx, target_idx]:
                continue

            strength = abs(edge_strengths[source_idx, target_idx])
            x_source, y_source = positions[source_idx]
            x_target, y_target = positions[target_idx]
            ax.plot(
                [x_source, x_target],
                [y_source, y_target],
                color="grey",
                alpha=min(1.0, max(0.15, strength * 2.5)),
                linewidth=0.4 + 3.0 * strength,
                zorder=0,
            )

    # 2. Draw Nodes
    ax.scatter(
        positions[:, 0],
        positions[:, 1],
        s=node_sizes,
        c=node_colors,
        alpha=0.90,
        edgecolors="white",
        linewidth=0.50,
        zorder=1,
    )

    # 3. Label only the most connected nodes. With ~500 stocks per frame, labelling
    # every one produces overlapping text that hides the structure it describes.
    labelled = set(np.argsort(node_sizes)[-max_labels:]) if len(tickers) > max_labels else set(range(len(tickers)))
    for index, name in enumerate(tickers):
        if index not in labelled:
            continue
        x, y = positions[index]
        
        # Calculate a slight offset based on the node size so labels don't overlap the dot
        # np.sqrt(node_sizes[index]) gives the approximate radius scaling factor
        radius_offset = 0.001 * np.sqrt(node_sizes[index]) if isinstance(node_sizes, np.ndarray) else 0.005
        
        ax.text(
            x + radius_offset + 0.002,  # Shift slightly to the right of the node
            y,                          # Align vertically with the center
            name,
            size=6.0,                   # Small, clean text size
            verticalalignment="center",
            horizontalalignment="left",
            zorder=2,                   # Ensure text sits cleanly on top of nodes/edges
            bbox=dict(
                facecolor="white", 
                edgecolor="none", 
                alpha=0.6,              # Semi-transparent background for readability
                boxstyle="round,pad=0.1"
            )
        )

    # Automatically expand the view margins slightly so text on the outer edges doesn't get clipped
    x_coords = positions[:, 0]
    y_coords = positions[:, 1]
    x_padding = np.ptp(x_coords) * 0.08 if len(x_coords) > 1 else 0.1
    y_padding = np.ptp(y_coords) * 0.05 if len(y_coords) > 1 else 0.1
    
    ax.set_xlim(x_coords.min() - x_padding, x_coords.max() + x_padding)
    ax.set_ylim(y_coords.min() - y_padding, y_coords.max() + y_padding)

    if color_by_sector:
        ax.legend(
            handles=build_sector_legend(color_ids),
            loc="center left",
            bbox_to_anchor=(1.0, 0.5),
            fontsize=10,
            title=legend_title,
            title_fontsize=12,
            frameon=False,
        )
    else:
        ax.text(1.01, 0.5, legend_title, transform=ax.transAxes, fontsize=12,
                verticalalignment="center")
    ax.set_title(title, fontsize=18, pad=20)

    frame_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(frame_path, dpi=figure_dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)

def export_gif(frame_paths: list[Path], gif_path: Path, duration_seconds: float) -> None:
    if not frame_paths:
        raise ValueError(f"No frames were created for {gif_path.name}.")

    frames = [imageio.imread(frame_path) for frame_path in frame_paths]
    imageio.mimsave(gif_path, frames, duration=duration_seconds, loop=0)
    print(f"Saved GIF: {gif_path}")

## Load Inputs

This cell loads the stock price panel and sector mapping once. Both downstream methods reuse these objects.


In [90]:
price_matrix = load_price_matrix(config.stock_data_dir, config.min_ticker_coverage)
ticker_to_sector = load_sector_mapping(config.sector_mapping_path)
global_mds_layout = build_mds_layout(price_matrix)

Loaded price matrix: 2,156 dates x 398 tickers
Loaded 190 ticker-sector mappings
Built global MDS layout for 398 tickers


## Calibrating the hyperparameters once

`GraphicalLassoCV` and `AffinityPropagation`'s default `preference` are both data-dependent: run per year, each year gets a different L1 penalty and a different preference (sklearn defaults preference to the median input similarity). Edge counts and cluster counts would then differ across years partly because the *estimator* changed, not because the market did.

So both are calibrated once on the pooled 2004-2024 sample and held fixed for every annual frame. Cross-year comparisons are then comparisons of the market under a constant estimator.


In [ ]:
pooled_returns = calculate_returns(price_matrix)

# One CV fit on the pooled sample picks the penalty; every annual frame then reuses it.
pooled_cv = GraphicalLassoCV().fit(standardize_returns(pooled_returns))
pooled_correlation = pooled_returns.corr().fillna(0.0).to_numpy()
off_diagonal = pooled_correlation[np.triu_indices_from(pooled_correlation, k=1)]

config = replace(
    config,
    graphical_lasso_alpha=float(pooled_cv.alpha_),
    affinity_preference=float(np.median(off_diagonal)),
)
print(f"Fixed graphical lasso alpha: {config.graphical_lasso_alpha:.4f}")
print(f"Fixed affinity propagation preference: {config.affinity_preference:.4f}")


# MDS Section: Graphical Lasso Network

This section estimates sparse conditional dependence networks with the Graphical Lasso at the **fixed** penalty calibrated above, and uses the global MDS coordinates to keep ticker positions stable across annual frames.

An edge shows that there is a correlation, in which a thicker edge means that there is even more of a correlation.
The L1 penalty applies to the precision (inverse covariance) matrix, shrinking weak relationships to exactly zero. What survives is a *partial* correlation: the dependency between two stocks conditional on all the others, which strips out the common market factor. Because the penalty is the same in every year, the resulting edge density is comparable across years - that number is written to `figures/network_density.csv`.

In [91]:
def estimate_partial_correlations(returns: pd.DataFrame, alpha: float,
                                  ridge: float = 0.10) -> np.ndarray:
    # ridge=0.10 is the smallest value that keeps every year solvable (2011 fails
    # below it). The relative ordering of yearly densities is unchanged from 0.01
    # to 0.10, so this affects conditioning rather than the result.
    standardized_values = standardize_returns(returns)
    empirical_correlation = np.corrcoef(standardized_values, rowvar=False)
    n_features = empirical_correlation.shape[0]
    empirical_correlation = ((1.0 - ridge) * empirical_correlation
                             + ridge * np.eye(n_features))
    _, precision_matrix = graphical_lasso(empirical_correlation, alpha=alpha, max_iter=200)
    diagonal_scale = np.sqrt(np.diag(precision_matrix))
    partial_correlations = precision_matrix / diagonal_scale[:, None] / diagonal_scale[None, :]
    np.fill_diagonal(partial_correlations, 0.0)
    return partial_correlations


def build_top_k_edge_matrix(partial_correlations: np.ndarray, edge_k: int) -> np.ndarray:
    n_tickers = partial_correlations.shape[0]
    edges = np.zeros((n_tickers, n_tickers), dtype=bool)

    for ticker_idx in range(n_tickers):
        strongest_neighbor_indices = np.argsort(np.abs(partial_correlations[ticker_idx]))[-edge_k:]
        edges[ticker_idx, strongest_neighbor_indices] = True

    np.fill_diagonal(edges, False)
    return edges | edges.T


def run_graphical_lasso_mds_animation(
    price_matrix: pd.DataFrame,
    ticker_to_sector: dict[str, str],
    mds_layout: dict[str, np.ndarray],
    config: AnalysisConfig,
) -> list[Path]:
    frame_paths: list[Path] = []
    density_rows: list[dict] = []
    frame_dir = config.output_dir / "mds_frames"

    for year in range(config.start_year, config.end_year + 1):
        start_year, window_prices = window_price_matrix(price_matrix, year, config.rolling_window_years)
        if len(window_prices) < config.min_window_observations:
            continue

        returns = calculate_returns(window_prices)
        if returns.shape[1] < config.min_window_tickers:
            continue

        partial_correlations = estimate_partial_correlations(returns, config.graphical_lasso_alpha)
        # Share of stock pairs with a surviving conditional dependency. Because alpha is
        # fixed, this IS comparable across years - unlike the drawn edges, which are
        # capped at top-k per node and so are constant by construction.
        off_diagonal = partial_correlations[np.triu_indices_from(partial_correlations, k=1)]
        density_rows.append({
            "year": year,
            "n_tickers": partial_correlations.shape[0],
            "n_days": len(returns),
            "edge_density": float(np.mean(np.abs(off_diagonal) > 1e-6)),
            "max_partial_corr": float(np.abs(off_diagonal).max()),
        })
        edges = build_top_k_edge_matrix(partial_correlations, config.graphical_lasso_edge_k)
        tickers = returns.columns.tolist()

        valid_tickers = [ticker for ticker in tickers if ticker in mds_layout]
        valid_indices = [tickers.index(ticker) for ticker in valid_tickers]
        if len(valid_tickers) < config.min_window_tickers:
            continue

        aligned_partial_correlations = partial_correlations[np.ix_(valid_indices, valid_indices)]
        aligned_edges = edges[np.ix_(valid_indices, valid_indices)]
        positions = np.array([mds_layout[ticker] for ticker in valid_tickers])
        node_sizes = 20 + 8 * aligned_edges.sum(axis=1)

        frame_path = frame_dir / f"{year}.png"
        render_network_frame(
            frame_path=frame_path,
            title=(f"S&P 500 Graphical Lasso Partial-Correlation Network ({year})   "
                   f"[{window_coverage(returns, year):.0%} of trading days observed]"),
            tickers=valid_tickers,
            positions=positions,
            edges=aligned_edges,
            edge_strengths=aligned_partial_correlations,
            node_sizes=node_sizes,
            ticker_to_sector=ticker_to_sector,
            figure_size=config.figure_size,
            figure_dpi=config.figure_dpi,
        )
        frame_paths.append(frame_path)
        print(f"Created MDS frame for {year}")

    density = pd.DataFrame(density_rows).set_index("year")
    density.to_csv(config.output_dir / "network_density.csv")
    print("\nNetwork density at fixed alpha:")
    print(density.round(4).to_string())
    return frame_paths


In [93]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*slogdet.*")
    
    mds_frame_paths = run_graphical_lasso_mds_animation(
            price_matrix=price_matrix,
            ticker_to_sector=ticker_to_sector,
            mds_layout=global_mds_layout,
            config=config,
    )

# Export your GIF cleanly
export_gif(
        frame_paths=mds_frame_paths,
        gif_path=config.output_dir / "stock_market_structure_mds.gif",
        duration_seconds=config.gif_duration_seconds,
)

/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)
/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/sklearn/covariance/_graph_lasso.py:199: ConvergenceWarning: graphical_lasso: did not converge after 100 iteration: dual gap: -1.298e-01
  warnings.warn(


Created MDS frame for 2004


/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)
/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/sklearn/covariance/_graph_lasso.py:199: ConvergenceWarning: graphical_lasso: did not converge after 100 iteration: dual gap: -1.111e-01
  warnings.warn(


Created MDS frame for 2005


/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)
/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/sklearn/covariance/_graph_lasso.py:199: ConvergenceWarning: graphical_lasso: did not converge after 100 iteration: dual gap: -5.026e-02
  warnings.warn(


Created MDS frame for 2006


... (24 outputs left out)

/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)
/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/sklearn/covariance/_graph_lasso.py:199: ConvergenceWarning: graphical_lasso: did not converge after 100 iteration: dual gap: -9.844e-02
  warnings.warn(


Created MDS frame for 2023


/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)
/Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/.venv/lib/python3.14/site-packages/sklearn/covariance/_graph_lasso.py:199: ConvergenceWarning: graphical_lasso: did not converge after 100 iteration: dual gap: 5.024e-03
  warnings.warn(


Created MDS frame for 2024
Saved GIF: /Users/ammarnagri/Documents/GitHub/Changes-in-Stock-Market-Structure/figures/stock_market_structure_mds.gif


# Affinity Propagation Section: Exemplar-Based Clusters

This section clusters stocks directly from the rolling correlation matrix using `AffinityPropagation`. Each cluster member is connected to its exemplar, creating a visualisation of market groups.

In [94]:
def build_affinity_propagation_edges(
    correlation_matrix: np.ndarray,
    damping: float,
    preference: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model = AffinityPropagation(
        affinity="precomputed",
        damping=damping,
        preference=preference,
        random_state=RANDOM_STATE,
    )
    model.fit(correlation_matrix)

    labels = model.labels_
    exemplar_indices = model.cluster_centers_indices_
    if exemplar_indices is None or len(exemplar_indices) == 0:
        raise RuntimeError("Affinity Propagation did not identify any exemplars.")

    n_tickers = correlation_matrix.shape[0]
    edges = np.zeros((n_tickers, n_tickers), dtype=bool)

    for ticker_idx in range(n_tickers):
        exemplar_idx = exemplar_indices[labels[ticker_idx]]
        if ticker_idx != exemplar_idx:
            edges[ticker_idx, exemplar_idx] = True
            edges[exemplar_idx, ticker_idx] = True

    return edges, exemplar_indices, labels


def run_affinity_propagation_animation(
    price_matrix: pd.DataFrame,
    ticker_to_sector: dict[str, str],
    mds_layout: dict[str, np.ndarray],
    config: AnalysisConfig,
) -> list[Path]:
    frame_paths: list[Path] = []
    frame_dir = config.output_dir / "affinity_propagation_frames"

    for year in range(config.start_year, config.end_year + 1):
        start_year, window_prices = window_price_matrix(price_matrix, year, config.rolling_window_years)
        if len(window_prices) < config.min_window_observations:
            continue

        returns = calculate_returns(window_prices)
        if returns.shape[1] < config.min_window_tickers:
            continue

        tickers = returns.columns.tolist()
        correlation_matrix = returns.corr().fillna(0.0).to_numpy()

        try:
            edges, exemplar_indices, cluster_labels = build_affinity_propagation_edges(
                correlation_matrix=correlation_matrix,
                damping=config.affinity_damping,
                preference=config.affinity_preference,
            )
        except RuntimeError as exc:
            print(f"Skipping {year}: {exc}")
            continue

        valid_tickers = [ticker for ticker in tickers if ticker in mds_layout]
        valid_indices = [tickers.index(ticker) for ticker in valid_tickers]
        if len(valid_tickers) < config.min_window_tickers:
            continue

        aligned_correlation_matrix = correlation_matrix[np.ix_(valid_indices, valid_indices)]
        aligned_edges = edges[np.ix_(valid_indices, valid_indices)]
        positions = np.array([mds_layout[ticker] for ticker in valid_tickers])
        node_sizes = 20 + 20 * aligned_edges.sum(axis=1)

        frame_path = frame_dir / f"{year}.png"
        render_network_frame(
            frame_path=frame_path,
            title=(f"S&P 500 Affinity Propagation Clusters ({year})   "
                   f"[{window_coverage(returns, year):.0%} of trading days observed]"),
            tickers=valid_tickers,
            positions=positions,
            edges=aligned_edges,
            edge_strengths=aligned_correlation_matrix,
            node_sizes=node_sizes,
            ticker_to_sector=ticker_to_sector,
            node_color_ids=cluster_labels[valid_indices],
            legend_title=f"{len(exemplar_indices)} clusters",
            figure_size=config.figure_size,
            figure_dpi=config.figure_dpi,
        )
        frame_paths.append(frame_path)
        print(f"Created Affinity Propagation frame for {year} with {len(exemplar_indices)} exemplars")

    return frame_paths


In [95]:
affinity_frame_paths = run_affinity_propagation_animation(
    price_matrix=price_matrix,
    ticker_to_sector=ticker_to_sector,
    mds_layout=global_mds_layout,
    config=config,
)
export_gif(
    frame_paths=affinity_frame_paths,
    gif_path=config.output_dir / "stock_market_structure_affinity_propagation.gif",
    duration_seconds=config.gif_duration_seconds,
)

Created Affinity Propagation frame for 2004 with 30 exemplars
Created Affinity Propagation frame for 2005 with 29 exemplars
Created Affinity Propagation frame for 2006 with 30 exemplars
Created Affinity Propagation frame for 2007 with 31 exemplars
Created Affinity Propagation frame for 2008 with 33 exemplars
Created Affinity Propagation frame for 2009 with 36 exemplars
Created Affinity Propagation frame for 2010 with 41 exemplars
Created Affinity Propagation frame for 2011 with 36 exemplars
Created Affinity Propagation frame for 2012 with 33 exemplars
Created Affinity Propagation frame for 2013 with 39 exemplars
Created Affinity Propagation frame for 2014 with 43 exemplars
Created Affinity Propagation frame for 2015 with 35 exemplars
Created Affinity Propagation frame for 2016 with 37 exemplars
Created Affinity Propagation frame for 2017 with 40 exemplars
Created Affinity Propagation frame for 2018 with 33 exemplars
Created Affinity Propagation frame for 2023 with 30 exemplars
Created 